In [1]:
# =========================================
# Zelle 1: Umgebung vorbereiten
# =========================================

# numexpr und bottleneck neu bauen — beide inkompatibel mit NumPy 2.x
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "numexpr", "bottleneck"],
    check=True
)
print("Fertig — bitte Kernel neu starten (Kernel → Restart)")

Fertig — bitte Kernel neu starten (Kernel → Restart)


In [2]:
# =========================================
# Zelle 1: Imports und Konstanten
# =========================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Pfade und Station
STATION_UUID = "e1aefc4e-3ca1-4018-8d91-455b69d35d41"
DATA_DIR     = Path("../data")
ML_DIR       = DATA_DIR / "ml"

print("Imports ok")
print(f"NumPy:  {np.__version__}")
print(f"Pandas: {pd.__version__}")

Imports ok
NumPy:  2.0.2
Pandas: 3.0.1


In [3]:
# =========================================
# Zelle 2: Preisdaten laden
# =========================================

# Rohdaten laden — nur unsere Station, Diesel als eigene Spalte
df_raw = pd.read_parquet(DATA_DIR / "tankstellen_preise.parquet")

df = df_raw[
    (df_raw["station_uuid"] == STATION_UUID) &
    (df_raw["diesel"].notna())
].copy()

df = df[["date", "diesel"]].copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Zeilen:   {len(df):,}")
print(f"Zeitraum: {df['date'].min().date()} → {df['date'].max().date()}")

Zeilen:   78,313
Zeitraum: 2017-06-09 → 2026-03-25


In [4]:
# =========================================
# Zelle 3: Tagesmittel berechnen
# =========================================

# Intraday-Schwankungen herausrechnen — wir arbeiten auf Tagesbasis
df["tag"] = df["date"].dt.date

df_tag = (
    df.groupby("tag")["diesel"]
    .mean()
    .reset_index()
    .rename(columns={"diesel": "mean_preis"})
)

df_tag["tag"] = pd.to_datetime(df_tag["tag"])
df_tag = df_tag.sort_values("tag").reset_index(drop=True)

print(f"Tage gesamt: {len(df_tag):,}")
print(f"Zeitraum:    {df_tag['tag'].min().date()} → {df_tag['tag'].max().date()}")
print(df_tag.head())

Tage gesamt: 3,182
Zeitraum:    2017-06-09 → 2026-03-25
         tag  mean_preis
0 2017-06-09    1.109000
1 2017-06-10    1.127889
2 2017-06-11    1.092333
3 2017-06-12    1.132571
4 2017-06-13    1.085667


In [5]:
# =========================================
# Zelle 4: Zielvariablen berechnen
# =========================================

# t0 = gestern (Basis), t1 = heute, t2 = morgen
# Modell A: mean(t1) - mean(t0) — wie verändert sich der Preis heute vs. gestern?
# Modell B: mean(t2) - mean(t0) — wie verändert sich der Preis morgen vs. gestern?

df_tag["mean_t0"] = df_tag["mean_preis"]
df_tag["mean_t1"] = df_tag["mean_preis"].shift(-1)  # nächster Tag
df_tag["mean_t2"] = df_tag["mean_preis"].shift(-2)  # übernächster Tag

df_tag["ziel_A"] = df_tag["mean_t1"] - df_tag["mean_t0"]
df_tag["ziel_B"] = df_tag["mean_t2"] - df_tag["mean_t0"]

# Letzte zwei Zeilen haben kein t1/t2 — rauswerfen
df_tag = df_tag.dropna(subset=["ziel_A", "ziel_B"]).reset_index(drop=True)

print(f"Tage nach Bereinigung: {len(df_tag):,}")
print(f"\nZielvariable A (mean_t1 - mean_t0):")
print(df_tag["ziel_A"].describe().round(4))
print(f"\nZielvariable B (mean_t2 - mean_t0):")
print(df_tag["ziel_B"].describe().round(4))

Tage nach Bereinigung: 3,180

Zielvariable A (mean_t1 - mean_t0):
count    3180.0000
mean        0.0004
std         0.0168
min        -0.1452
25%        -0.0089
50%         0.0002
75%         0.0094
max         0.1140
Name: ziel_A, dtype: float64

Zielvariable B (mean_t2 - mean_t0):
count    3180.0000
mean        0.0007
std         0.0220
min        -0.1155
25%        -0.0111
50%         0.0000
75%         0.0119
max         0.2110
Name: ziel_B, dtype: float64


In [6]:
# =========================================
# Zelle 5: Externe Features laden
# =========================================

# Brent-Tagespreise
brent = pd.read_csv(DATA_DIR / "brent_futures_daily.csv", parse_dates=["period"])
brent = brent.rename(columns={"period": "tag"})
brent = brent.sort_values("tag").reset_index(drop=True)

# EUR/USD-Tageskurs
eur_usd = pd.read_csv(DATA_DIR / "eur_usd_rate.csv", parse_dates=["period"])
eur_usd = eur_usd.rename(columns={"period": "tag"})
eur_usd = eur_usd.sort_values("tag").reset_index(drop=True)

# Feiertage NRW
feiertage = pd.read_csv(DATA_DIR / "feiertage.csv", parse_dates=["datum"])
feiertage_nrw = feiertage[feiertage["bundesland_kuerzel"].str.contains("NW", na=False)].copy()

# Schulferien NRW
schulferien = pd.read_csv(DATA_DIR / "schulferien.csv")
schulferien = schulferien[schulferien["bundesland_code"] == "DE-NW"].copy()
schulferien["datum_start"] = pd.to_datetime(schulferien["datum_start"])
schulferien["datum_ende"]  = pd.to_datetime(schulferien["datum_ende"])

print("Brent:")
print(f"  Spalten:  {brent.columns.tolist()}")
print(f"  Zeitraum: {brent['tag'].min().date()} → {brent['tag'].max().date()}")
print("\nEUR/USD:")
print(f"  Spalten:  {eur_usd.columns.tolist()}")
print(f"  Zeitraum: {eur_usd['tag'].min().date()} → {eur_usd['tag'].max().date()}")

Brent:
  Spalten:  ['tag', 'brent_futures_usd']
  Zeitraum: 2014-01-02 → 2026-03-25

EUR/USD:
  Spalten:  ['tag', 'eur_usd']
  Zeitraum: 1999-01-04 → 2026-03-25


In [ ]:
# =========================================
# Zelle 6: Features in df_tag mergen
# =========================================

# Spalten entfernen falls Zelle nochmal ausgeführt wird
merge_spalten = [
    "brent_futures_usd", "eur_usd", "brent_eur",
    "ist_feiertag", "ist_feiertag_t1", "ist_feiertag_t2",
    "ist_schulferien", "ist_schulferien_t1", "ist_schulferien_t2",
    "wochentag", "wochentag_t1", "wochentag_t2"
]
df_tag = df_tag.drop(columns=[c for c in merge_spalten if c in df_tag.columns])

# Brent und EUR/USD per Datum joinen
df_tag = df_tag.merge(brent,   on="tag", how="left")
df_tag = df_tag.merge(eur_usd, on="tag", how="left")

# Fehlende Brent/EUR/USD Werte vorwärts füllen (Wochenenden, Feiertage)
df_tag["brent_futures_usd"] = df_tag["brent_futures_usd"].ffill()
df_tag["eur_usd"]           = df_tag["eur_usd"].ffill()

# Brent in EUR umrechnen
df_tag["brent_eur"] = df_tag["brent_futures_usd"] / df_tag["eur_usd"]

# Hilfsspalten für t1 und t2
df_tag["tag_t1"] = df_tag["tag"] + pd.Timedelta(days=1)
df_tag["tag_t2"] = df_tag["tag"] + pd.Timedelta(days=2)

# Feiertag NRW — t0, t1, t2
feiertage_set = set(feiertage_nrw["datum"].dt.date)
df_tag["ist_feiertag"]    = df_tag["tag"].dt.date.isin(feiertage_set).astype(int)
df_tag["ist_feiertag_t1"] = df_tag["tag_t1"].dt.date.isin(feiertage_set).astype(int)
df_tag["ist_feiertag_t2"] = df_tag["tag_t2"].dt.date.isin(feiertage_set).astype(int)

# Schulferien NRW — t0, t1, t2
def ist_schulferien(tag):
    return int(any(
        (row["datum_start"].date() <= tag <= row["datum_ende"].date())
        for _, row in schulferien.iterrows()
    ))

df_tag["ist_schulferien"]    = df_tag["tag"].dt.date.apply(ist_schulferien)
df_tag["ist_schulferien_t1"] = df_tag["tag_t1"].dt.date.apply(ist_schulferien)
df_tag["ist_schulferien_t2"] = df_tag["tag_t2"].dt.date.apply(ist_schulferien)

# Wochentag — t0, t1, t2
df_tag["wochentag"]    = df_tag["tag"].dt.dayofweek
df_tag["wochentag_t1"] = df_tag["tag_t1"].dt.dayofweek
df_tag["wochentag_t2"] = df_tag["tag_t2"].dt.dayofweek

# Hilfsspalten wieder entfernen
df_tag = df_tag.drop(columns=["tag_t1", "tag_t2"])

print(f"Shape: {df_tag.shape}")
print(f"NaN gesamt: {df_tag.isnull().sum().sum()}")

In [ ]:
# =========================================
# Zelle 7: Lag-Features berechnen
# =========================================

# Lags aus bereits gemergten Brent/EUR/USD Spalten berechnen
df_tag["brent_lag1"] = df_tag["brent_futures_usd"].shift(1)
df_tag["brent_lag2"] = df_tag["brent_futures_usd"].shift(2)
df_tag["brent_lag3"] = df_tag["brent_futures_usd"].shift(3)

df_tag["brent_delta1"] = df_tag["brent_lag1"] - df_tag["brent_lag2"]
df_tag["brent_delta2"] = df_tag["brent_lag2"] - df_tag["brent_lag3"]

df_tag["eur_usd_lag1"] = df_tag["eur_usd"].shift(1)
df_tag["eur_usd_lag2"] = df_tag["eur_usd"].shift(2)
df_tag["eur_usd_lag3"] = df_tag["eur_usd"].shift(3)

df_tag["eur_usd_delta1"] = df_tag["eur_usd_lag1"] - df_tag["eur_usd_lag2"]
df_tag["eur_usd_delta2"] = df_tag["eur_usd_lag2"] - df_tag["eur_usd_lag3"]

df_tag["brent_eur_lag1"]   = df_tag["brent_eur"].shift(1)
df_tag["brent_eur_delta1"] = df_tag["brent_eur_lag1"] - df_tag["brent_eur"].shift(2)

# Rolling-Mittel Diesel — Preisniveau der letzten Wochen
df_tag["roll7d"]  = df_tag["mean_preis"].rolling(7,  min_periods=1).mean()
df_tag["roll30d"] = df_tag["mean_preis"].rolling(30, min_periods=1).mean()

# NaN durch Lags entfernen
df_tag = df_tag.dropna().reset_index(drop=True)

print(f"Shape nach Lags: {df_tag.shape}")
print(f"NaN gesamt:      {df_tag.isnull().sum().sum()}")

Shape nach Lags: (2168, 41)
NaN gesamt:      0


In [ ]:
# =========================================
# Zelle 8: Externe Effekte und Energiesteuer laden und mergen
# =========================================

# Binäre Event-Flags — Lockdown, Niedrigwasser
externe_effekte = pd.read_csv(DATA_DIR / "externe_effekte.csv", parse_dates=["date"])
externe_effekte = externe_effekte.rename(columns={"date": "tag"})

# Energiesteuer und Tankrabatt
energiesteuer = pd.read_csv(DATA_DIR / "energiesteuer.csv", parse_dates=["date"])
energiesteuer = energiesteuer.rename(columns={"date": "tag"})

# Spalten entfernen falls Zelle nochmal ausgeführt wird
effekt_spalten     = ["ist_lockdown", "ist_niedrigwasser"]
energie_spalten    = ["ist_tankrabatt", "energiesteuer_diesel", "energiesteuer_benzin"]
bereits_vorhanden  = [c for c in effekt_spalten + energie_spalten if c in df_tag.columns]
df_tag = df_tag.drop(columns=bereits_vorhanden, errors="ignore")

df_tag = df_tag.merge(externe_effekte, on="tag", how="left")
df_tag = df_tag.merge(energiesteuer,   on="tag", how="left")

# Fehlende Werte mit 0 füllen — kein Event bedeutet 0
df_tag["ist_lockdown"]         = df_tag["ist_lockdown"].fillna(0).astype(int)
df_tag["ist_niedrigwasser"]    = df_tag["ist_niedrigwasser"].fillna(0).astype(int)
df_tag["ist_tankrabatt"]       = df_tag["ist_tankrabatt"].fillna(0).astype(int)
df_tag["energiesteuer_diesel"] = df_tag["energiesteuer_diesel"].ffill()
df_tag = df_tag.drop(columns=["energiesteuer_benzin"])

print(f"Shape: {df_tag.shape}")
print(f"NaN gesamt: {df_tag.isnull().sum().sum()}")
print(df_tag.columns.tolist())


Shape: (2168, 41)
NaN gesamt: 0
['tag', 'mean_preis', 'mean_t0', 'mean_t1', 'mean_t2', 'ziel_A', 'ziel_B', 'brent_futures_usd_x', 'eur_usd_x', 'brent_lag1', 'brent_lag2', 'brent_lag3', 'brent_delta1', 'brent_delta2', 'eur_usd_lag1', 'eur_usd_lag2', 'eur_usd_lag3', 'eur_usd_delta1', 'eur_usd_delta2', 'brent_eur_lag1', 'brent_eur_delta1', 'roll7d', 'roll30d', 'brent_futures_usd_y', 'eur_usd_y', 'brent_futures_usd', 'eur_usd', 'brent_eur', 'ist_feiertag', 'ist_feiertag_t1', 'ist_feiertag_t2', 'ist_schulferien', 'ist_schulferien_t1', 'ist_schulferien_t2', 'wochentag', 'wochentag_t1', 'wochentag_t2', 'ist_lockdown', 'ist_niedrigwasser', 'energiesteuer_diesel', 'ist_tankrabatt']


In [ ]:
# =========================================
# Zelle 9: Feature-Spalten definieren
# =========================================

# Spalten die nicht ins Modell gehen
NICHT_FEATURES = ["tag", "mean_preis", "mean_t0", "mean_t1", "mean_t2", "ziel_A", "ziel_B"]

# Alle übrigen Spalten sind Features
feature_cols = [c for c in df_tag.columns if c not in NICHT_FEATURES]

print(f"Anzahl Features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

Anzahl Features: 34
Features: ['brent_futures_usd_x', 'eur_usd_x', 'brent_lag1', 'brent_lag2', 'brent_lag3', 'brent_delta1', 'brent_delta2', 'eur_usd_lag1', 'eur_usd_lag2', 'eur_usd_lag3', 'eur_usd_delta1', 'eur_usd_delta2', 'brent_eur_lag1', 'brent_eur_delta1', 'roll7d', 'roll30d', 'brent_futures_usd_y', 'eur_usd_y', 'brent_futures_usd', 'eur_usd', 'brent_eur', 'ist_feiertag', 'ist_feiertag_t1', 'ist_feiertag_t2', 'ist_schulferien', 'ist_schulferien_t1', 'ist_schulferien_t2', 'wochentag', 'wochentag_t1', 'wochentag_t2', 'ist_lockdown', 'ist_niedrigwasser', 'energiesteuer_diesel', 'ist_tankrabatt']
